# YOLOv11-Nano Fine-Tuning: Otomotiv Parça Tespiti

**Proje:** ZeroHalt - Araç Parça Tanıma Sistemi  
**Model:** YOLOv11n (nano)  
**Sınıflar:** `alternator`, `brake_disc`, `engine_control`, `hydraulic_filter`, `piston`

---

## Pipeline Özeti

```
1. Ortam Kurulumu
2. Veri Keşfi & Analiz
3. Veri Artırma (Augmentation) & Hazırlama
4. Annotation Rehberi (LabelImg / otomatik)
5. YOLO Veri Seti Formatına Dönüşüm
6. Dataset YAML Oluşturma
7. Model Fine-Tuning
8. Eğitim Metrikleri & Görselleştirme
9. Model Değerlendirme (Validation)
10. Inference & Test
11. Model Export (ONNX / TensorRT)
```

## 1. Ortam Kurulumu

In [ ]:
# Gerekli kütüphaneleri kur
!pip install ultralytics==8.3.0 --quiet
!pip install Pillow==10.4.0 --quiet
!pip install albumentations==1.4.0 --quiet
!pip install opencv-python==4.10.0.84 --quiet
!pip install matplotlib seaborn pandas numpy --quiet
!pip install roboflow --quiet          # Roboflow entegrasyonu (opsiyonel)
!pip install labelimg --quiet          # Manuel annotation aracı (opsiyonel)
!pip install icrawler --quiet          # Ek veri toplama için
print('Kurulum tamamlandı.')

In [ ]:
import os
import shutil
import random
import json
import math
import warnings
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from pathlib import Path
from PIL import Image, ImageFilter, ImageEnhance
from collections import defaultdict
import albumentations as A

import torch
from ultralytics import YOLO

# Tekrarlanabilirlik
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA mevcut: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ─── TEMEL KONFİGÜRASYON ───────────────────────────────────────────────────
BASE_DIR      = Path('.')                        # notebook'un bulunduğu klasör
RAW_DATA_DIR  = BASE_DIR / 'parcalar'            # ham görsel klasörü
DATASET_DIR   = BASE_DIR / 'dataset'             # işlenmiş YOLO veri seti
RUNS_DIR      = BASE_DIR / 'runs'

# Sınıf tanımları (klasör adı → temiz isim → class_id)
CLASS_MAP = {
    'alternator':     {'id': 0, 'label': 'alternator'},
    'brake disc':     {'id': 1, 'label': 'brake_disc'},
    'engine control': {'id': 2, 'label': 'engine_control'},
    'hydroic filter': {'id': 3, 'label': 'hydraulic_filter'},
    'piston':         {'id': 4, 'label': 'piston'},
}

CLASS_NAMES = [v['label'] for v in sorted(CLASS_MAP.values(), key=lambda x: x['id'])]
NUM_CLASSES  = len(CLASS_NAMES)

# Eğitim hiper-parametreleri (küçük veri setine göre ayarlı)
IMG_SIZE    = 640
BATCH_SIZE  = 8       # az veri → küçük batch, daha çok gradyan adımı
EPOCHS      = 200
LR0         = 0.01
LRF         = 0.001
MOMENTUM    = 0.937
WEIGHT_DECAY = 0.0005
PATIENCE    = 50      # early stopping (küçük veride sabırlı ol)
WORKERS     = 4

# Veri bölme oranları (test YOK — veri çok az; sadece train + val)
# Val yalnızca ORİJİNAL görsellerden oluşur, augment edilmez.
VAL_RATIO   = 0.25

# Augmentation: SADECE nokta/serpme filtresi, veriyi ~2 katına çıkar.
# Her orijinal train görseli için kaç augment kopya üretilecek (1 → toplam 2x).
AUG_PER_IMAGE = 1

# Desteklenen görsel uzantıları (jpg ve webp karışık veri için)
SUPPORTED_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.tif'}


def read_image_rgb(img_path):
    """Görseli formattan bağımsız (jpg/jpeg/png/webp/bmp/tiff) RGB numpy dizisi okur.

    Önce Pillow ile dener (webp / animasyonlu webp dahil),
    başarısız olursa OpenCV'ye düşer.
    """
    from PIL import Image as _Image
    try:
        with _Image.open(img_path) as im:
            if getattr(im, 'is_animated', False):
                im.seek(0)
            return np.array(im.convert('RGB'))
    except Exception as e_pil:
        bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if bgr is None:
            raise IOError(f'Görsel okunamadı ({Path(img_path).name}): {e_pil}')
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


print(f'Sınıflar: {CLASS_NAMES}')
print(f'Augment kopya/orijinal: {AUG_PER_IMAGE} (toplam ~{AUG_PER_IMAGE + 1}x)')
print(f'Desteklenen uzantılar: {sorted(SUPPORTED_EXT)}')

## 2. Veri Keşfi & Analiz

In [ ]:
# Ham veri istatistikleri (SUPPORTED_EXT config hücresinde tanımlandı)
stats = []
for folder_name, info in CLASS_MAP.items():
    folder = RAW_DATA_DIR / folder_name
    images = [f for f in folder.iterdir() if f.suffix.lower() in SUPPORTED_EXT]
    widths, heights = [], []
    for img_path in images:
        try:
            with Image.open(img_path) as img:
                widths.append(img.width)
                heights.append(img.height)
        except Exception:
            pass
    stats.append({
        'class':        info['label'],
        'count':        len(images),
        'avg_width':    int(np.mean(widths))  if widths  else 0,
        'avg_height':   int(np.mean(heights)) if heights else 0,
        'after_augment': len(images) * (AUG_PER_IMAGE + 1),
    })

df_stats = pd.DataFrame(stats)
print(df_stats.to_string(index=False))
print(f'\nToplam ham görsel: {df_stats["count"].sum()}')

In [ ]:
# Sınıf dağılımı grafiği
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']

axes[0].bar(df_stats['class'], df_stats['count'], color=colors)
axes[0].set_title('Ham Görsel Sayısı (Sınıf Başına)', fontsize=13)
axes[0].set_ylabel('Görsel Sayısı')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(df_stats['class'], df_stats['after_augment'], color=colors, alpha=0.7)
axes[1].set_title(f'Augment Sonrası (~{AUG_PER_IMAGE + 1}x)', fontsize=13)
axes[1].set_ylabel('Toplam Görsel')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('data_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Örnek görselleri önizle
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(12, NUM_CLASSES * 3))
fig.suptitle('Sınıf Başına Örnek Görseller', fontsize=15, y=1.01)

for row, (folder_name, info) in enumerate(CLASS_MAP.items()):
    folder = RAW_DATA_DIR / folder_name
    images = [f for f in folder.iterdir() if f.suffix.lower() in SUPPORTED_EXT]
    samples = random.sample(images, min(3, len(images)))
    for col, img_path in enumerate(samples):
        ax = axes[row][col]
        img = read_image_rgb(img_path)
        ax.imshow(img)
        ax.set_title(f'{info["label"]}\n{img.shape[1]}x{img.shape[0]}', fontsize=9)
        ax.axis('off')
    for col in range(len(samples), 3):
        axes[row][col].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Ek Veri Toplama (İsteğe Bağlı)

Az veri varsa `icrawler` ile web'den ek görsel indirilebilir.  
Bu adımı atlamak için hücreyi çalıştırmayın.

In [ ]:
CRAWL_ENABLED = False   # ← True yaparsanız web'den ek görsel indirilir

if CRAWL_ENABLED:
    from icrawler.builtin import GoogleImageCrawler, BingImageCrawler

    SEARCH_QUERIES = {
        'alternator':     ['car alternator part', 'automotive alternator isolated'],
        'brake disc':     ['brake disc rotor', 'car brake disk close up'],
        'engine control': ['engine control unit ECU', 'car ECU module'],
        'hydroic filter': ['hydraulic filter part', 'hydralic oil filter'],
        'piston':         ['engine piston part', 'automotive piston isolated'],
    }

    CRAWL_PER_QUERY = 30  # sorgu başına indirilecek görsel

    for folder_name, queries in SEARCH_QUERIES.items():
        save_dir = RAW_DATA_DIR / folder_name / 'crawled'
        save_dir.mkdir(parents=True, exist_ok=True)
        for query in queries:
            print(f'  İndiriliyor: {query}')
            crawler = BingImageCrawler(
                storage={'root_dir': str(save_dir)},
                feeder_threads=2,
                parser_threads=2,
                downloader_threads=4,
            )
            crawler.crawl(keyword=query, max_num=CRAWL_PER_QUERY, min_size=(200, 200))
    print('Web'den veri toplama tamamlandı.')
else:
    print('Web taraması devre dışı. CRAWL_ENABLED=True yaparak aktifleştirebilirsiniz.')

## 4. Annotation Rehberi

YOLO object detection için her görsele **bounding box** etiketi gerekir.  
İki seçenek sunulmaktadır:

### Seçenek A — LabelImg (Manuel)
```bash
pip install labelimg
labelimg parcalar/<sinif_adi>
```
Format olarak **YOLO** seçin. Her görsel için etiket `parcalar/<sinif_adi>/labels/` altına kaydedilir.

### Seçenek B — Roboflow (Yarı Otomatik)
1. [roboflow.com](https://roboflow.com) üzerinde ücretsiz proje açın  
2. Görselleri yükleyin, sınıfları tanımlayın  
3. Roboflow'un Smart Polygon aracıyla hızlıca etiketleyin  
4. YOLO v8/v11 formatında dışa aktarın, API key'i aşağıya girin

### Seçenek C — Otomatik Annotation (Bu Notebook)
Az veriyle çalışırken **tüm görsel = tek bounding box** yaklaşımı kullanılabilir.  
Görsel içindeki nesne büyük çoğunluğu kaplıyorsa bu yeterlidir.  
Aşağıdaki hücre bu "whole-image bbox" etiketini otomatik üretir.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# ANNOTATION STRATEJİSİ SEÇİN
#   'auto'     → tüm görsel alanını tek bbox olarak etiketler (hızlı başlangıç)
#   'roboflow' → Roboflow'dan indirilen etiketleri kullanır
#   'manual'   → LabelImg ile oluşturulan etiketler klasörde hazır olmalı
# ──────────────────────────────────────────────────────────────────────────────
ANNOTATION_MODE = 'auto'

ROBOFLOW_API_KEY    = ''   # Roboflow API key (mode='roboflow' ise doldurun)
ROBOFLOW_WORKSPACE  = ''
ROBOFLOW_PROJECT    = ''
ROBOFLOW_VERSION    = 1

print(f'Annotation modu: {ANNOTATION_MODE}')

In [ ]:
if ANNOTATION_MODE == 'roboflow' and ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version  = project.version(ROBOFLOW_VERSION)
    dataset  = version.download('yolov8', location=str(DATASET_DIR))
    print(f'Roboflow veri seti indirildi: {DATASET_DIR}')
    SKIP_MANUAL_SPLIT = True
else:
    SKIP_MANUAL_SPLIT = False
    print('Manuel / otomatik annotation modu aktif.')

## 5. Augmentation Pipeline & Veri Hazırlama

In [ ]:
# ─── KÜÇÜK VERİ İÇİN YOĞUN AUGMENTATION ──────────────────────────────────────
# Klasör başına 4-8 görsel var. Genellemeyi artırmak için hem geometrik/renk
# dönüşümleri hem de 'nokta nokta' (salt&pepper, dot-grid, speckle, dropout)
# desenleri uyguluyoruz. Bu desenler modelin parçayı arka plan gürültüsünden
# bağımsız tanımasını sağlar.

import numpy as _np


def aug_salt_pepper(image, **kwargs):
    """Tuz-biber gürültüsü: görsele rastgele beyaz/siyah noktalar serper."""
    img = image.copy()
    h, w = img.shape[:2]
    amount = _np.random.uniform(0.01, 0.05)   # piksellerin %1-5'i
    n = int(amount * h * w)
    # Tuz (beyaz)
    ys = _np.random.randint(0, h, n); xs = _np.random.randint(0, w, n)
    img[ys, xs] = 255
    # Biber (siyah)
    ys = _np.random.randint(0, h, n); xs = _np.random.randint(0, w, n)
    img[ys, xs] = 0
    return img


def aug_dot_grid(image, **kwargs):
    """Düzenli nokta-ızgara deseni (senin tarif ettiğin 'nokta nokta' filtre)."""
    img = image.copy()
    h, w = img.shape[:2]
    spacing = _np.random.choice([10, 14, 18, 22])   # nokta aralığı
    radius  = _np.random.choice([1, 2, 3])           # nokta yarıçapı
    color   = int(_np.random.choice([0, 255]))       # siyah veya beyaz nokta
    alpha   = _np.random.uniform(0.4, 0.9)           # saydamlık
    overlay = img.copy()
    for y in range(spacing, h, spacing):
        for x in range(spacing, w, spacing):
            cv2.circle(overlay, (x, y), int(radius), (color, color, color), -1)
    return cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)


def aug_speckle(image, **kwargs):
    """Rastgele renkli serpme (spatter) - dağınık noktacıklar."""
    img = image.copy()
    h, w = img.shape[:2]
    n = int(_np.random.uniform(0.005, 0.02) * h * w)
    for _ in range(n):
        x, y = _np.random.randint(0, w), _np.random.randint(0, h)
        r = _np.random.randint(1, 3)
        col = tuple(int(c) for c in _np.random.randint(0, 256, 3))
        cv2.circle(img, (x, y), r, col, -1)
    return img


def aug_pixel_dropout(image, **kwargs):
    """Rastgele pikselleri sıfırlar (dağınık delik/nokta efekti)."""
    img = image.copy()
    mask = _np.random.random(img.shape[:2]) < _np.random.uniform(0.02, 0.08)
    img[mask] = 0
    return img


def aug_coarse_dropout(image, **kwargs):
    """Görsele rastgele dikdörtgen delikler açar (Cutout / CoarseDropout)."""
    img = image.copy()
    h, w = img.shape[:2]
    for _ in range(_np.random.randint(3, 8)):
        ch, cw = _np.random.randint(10, 40), _np.random.randint(10, 40)
        y = _np.random.randint(0, max(1, h - ch)); x = _np.random.randint(0, max(1, w - cw))
        img[y:y+ch, x:x+cw] = int(_np.random.choice([0, 127, 255]))
    return img


augment_pipeline = A.Compose([
    # SADECE 'nokta nokta' serpme/dropout efektlerinden biri uygulanır.
    # NOT: Boyutlandırma yok — geometri korunur; kareye letterbox yazma anında yapılır.
    A.OneOf([
        A.Lambda(name='salt_pepper',    image=aug_salt_pepper),
        A.Lambda(name='dot_grid',       image=aug_dot_grid),
        A.Lambda(name='speckle',        image=aug_speckle),
        A.Lambda(name='pixel_dropout',  image=aug_pixel_dropout),
        A.Lambda(name='coarse_dropout', image=aug_coarse_dropout),
    ], p=1.0),
])

print('Augmentation pipeline hazır (yalnızca nokta/serpme filtresi, boyut korunur).')

In [ ]:
# Augmentation önizleme — bir sınıftan örnek
sample_folder = RAW_DATA_DIR / 'piston'
sample_imgs   = [f for f in sample_folder.iterdir() if f.suffix.lower() in SUPPORTED_EXT]
src_img       = read_image_rgb(sample_imgs[0])
src_resized   = cv2.resize(src_img, (IMG_SIZE, IMG_SIZE))

# 1) Tam pipeline'dan rastgele varyasyonlar
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Tam Augmentation Pipeline — Rastgele Varyasyonlar (piston)', fontsize=14)
for ax in axes.flatten():
    augmented = augment_pipeline(image=src_img)['image']
    ax.imshow(augmented)
    ax.axis('off')
plt.tight_layout()
plt.savefig('augmentation_preview.png', dpi=150, bbox_inches='tight')
plt.show()

# 2) 'Nokta nokta' efektlerini tek tek göster
dot_effects = [
    ('Orijinal',        lambda im: im),
    ('Salt & Pepper',   aug_salt_pepper),
    ('Dot Grid',        aug_dot_grid),
    ('Speckle',         aug_speckle),
    ('Pixel Dropout',   aug_pixel_dropout),
    ('Coarse Dropout',  aug_coarse_dropout),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Nokta / Serpme Efektleri (küçük veri için)', fontsize=14)
for ax, (title, fn) in zip(axes.flatten(), dot_effects):
    ax.imshow(fn(src_resized.copy()))
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.savefig('dot_effects_preview.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. YOLO Veri Seti Oluşturma

In [ ]:
def make_yolo_label(class_id, cx, cy, bw, bh) -> str:
    """Tek bir YOLO etiket satırı: 'class cx cy w h' (normalize 0-1)."""
    return f'{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n'


def auto_bbox_norm(img_rgb):
    """Düz/ürün fotoğrafında objenin SIKI sınırlayıcı kutusunu otomatik bulur.

    Yaklaşım: köşelerden arka plan rengini tahmin et, arka plana yakın pikselleri
    maskele, kalan (obje) bölgenin sınırlayıcı kutusunu döndür. Normalize (cx,cy,w,h).
    Başarısız olursa tüm görseli kutu kabul eder.
    """
    h, w = img_rgb.shape[:2]
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    # Köşe medyanından arka plan parlaklığını tahmin et
    k = max(2, min(h, w) // 20)
    corners = np.concatenate([
        gray[:k, :k].ravel(), gray[:k, -k:].ravel(),
        gray[-k:, :k].ravel(), gray[-k:, -k:].ravel(),
    ])
    bg = np.median(corners)

    # Arka plandan yeterince farklı pikseller = ön plan (obje)
    diff = np.abs(gray.astype(np.int16) - bg)
    mask = (diff > 25).astype(np.uint8) * 255

    # Gürültüyü temizle
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  np.ones((5, 5), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))

    ys, xs = np.where(mask > 0)
    # Ön plan çok küçük/çok büyükse güvenilmez → tüm görsel
    if len(xs) < 0.02 * h * w or len(xs) > 0.99 * h * w:
        return (0.5, 0.5, 0.98, 0.98)

    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()
    pad = 0.02   # kutuya küçük pay ekle
    cx = ((x1 + x2) / 2) / w
    cy = ((y1 + y2) / 2) / h
    bw = min(1.0, (x2 - x1) / w + pad)
    bh = min(1.0, (y2 - y1) / h + pad)
    return (cx, cy, bw, bh)


def get_labels_for_image(img_path, img_rgb, class_id):
    """Görsel için YOLO etiketleri döndürür: [(cls, cx, cy, w, h), ...].

    Manuel/LabelImg etiketi varsa onu kullanır; yoksa otomatik sıkı kutu üretir.
    """
    label_path = img_path.parent / 'labels' / (img_path.stem + '.txt')
    if label_path.exists():
        labels = []
        for line in label_path.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) == 5:
                c, cx, cy, bw, bh = parts
                labels.append((int(c), float(cx), float(cy), float(bw), float(bh)))
        if labels:
            return labels
    cx, cy, bw, bh = auto_bbox_norm(img_rgb)
    return [(class_id, cx, cy, bw, bh)]


def collect_raw_images(class_map: dict, raw_dir: Path) -> dict:
    """Ham görsel yollarını sınıf bazında toplar (jpg, jpeg, png, webp, bmp, tiff)."""
    result = defaultdict(list)
    for folder_name, info in class_map.items():
        folder = raw_dir / folder_name
        for f in folder.rglob('*'):
            # 'labels' alt klasörünü ve desteklenmeyen uzantıları atla
            if f.is_file() and f.suffix.lower() in SUPPORTED_EXT and 'labels' not in f.parts:
                result[info['id']].append(f)
    return result


print('Yardımcı fonksiyonlar tanımlandı (auto-bbox + letterbox dahil).')

In [ ]:
if not SKIP_MANUAL_SPLIT:
    # Çıkış dizinleri (test YOK)
    for split in ['train', 'val']:
        for sub in ['images', 'labels']:
            (DATASET_DIR / split / sub).mkdir(parents=True, exist_ok=True)

    raw_images  = collect_raw_images(CLASS_MAP, RAW_DATA_DIR)
    aug_counter = defaultdict(int)
    written     = defaultdict(int)   # split → count

    # Görseller ORİJİNAL boyutta kaydedilir (farklı boyutlar sorun değil; YOLO eğitimde
    # letterbox ile imgsz'e getirir, oranı bozmaz). Sadece aşırı büyük görseller sınırlanır.
    MAX_SIDE = 1280

    def _write_samples(split_name, class_label, samples):
        """Örnekleri orijinal en-boy oranıyla .jpg olarak yazar; etiketler normalize."""
        for idx, (img_np, labels) in enumerate(samples):
            stem    = f'{class_label}_{split_name}_{idx:04d}'
            img_out = DATASET_DIR / split_name / 'images' / f'{stem}.jpg'
            lbl_out = DATASET_DIR / split_name / 'labels' / f'{stem}.txt'
            out = img_np
            h, w = out.shape[:2]
            m = max(h, w)
            if m > MAX_SIDE:                         # oranı koruyarak küçült
                s = MAX_SIDE / m
                out = cv2.resize(out, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
            cv2.imwrite(str(img_out), cv2.cvtColor(out, cv2.COLOR_RGB2BGR),
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            lbl_out.write_text(''.join(make_yolo_label(*lb) for lb in labels))
            written[split_name] += 1

    for class_id, img_paths in raw_images.items():
        class_label = CLASS_NAMES[class_id]

        # 1) Tüm ORİJİNAL görselleri oku + etiketlerini üret (auto sıkı kutu veya manuel)
        originals = []   # (np_image, labels_list)
        for img_path in img_paths:
            try:
                img_np = read_image_rgb(img_path)
            except Exception as e:
                print(f'  ⚠ Açılamadı: {img_path.name} — {e}')
                continue
            labels = get_labels_for_image(img_path, img_np, class_id)
            originals.append((img_np, labels))

        n = len(originals)
        if n == 0:
            print(f'[{class_label}] ⚠ görsel bulunamadı, atlanıyor')
            continue
        random.shuffle(originals)

        # 2) VERİ SIZINTISINI ÖNLE: önce orijinalleri train/val'e böl, SONRA sadece train'i augment et.
        #    Her sınıf için en az 1 orijinal val'e ayrılır. Val asla augment edilmez.
        n_val   = max(1, int(round(n * VAL_RATIO))) if n >= 2 else 0
        n_train = max(1, n - n_val)

        val_orig   = originals[:n_val]
        train_orig = originals[n_val:]
        if not train_orig:                      # n == 1 durumunda güvenlik
            train_orig, val_orig = originals, []

        print(f'[{class_label}] {n} orijinal → train {len(train_orig)} | val {len(val_orig)}')

        # 3) Sadece TRAIN setini augment et: her orijinale AUG_PER_IMAGE kopya (toplam ~2x)
        #    Nokta filtresi geometriyi değiştirmez → etiketler aynı kalır.
        train_samples = list(train_orig)        # orijinaller de dahil
        for src_np, src_labels in train_orig:
            for _ in range(AUG_PER_IMAGE):
                aug_np = augment_pipeline(image=src_np)['image']
                train_samples.append((aug_np, src_labels))
        aug_counter[class_label] = len(train_samples) - len(train_orig)
        print(f'    +{aug_counter[class_label]} augmented train görseli (toplam train {len(train_samples)})')

        # 4) Val SADECE orijinal görsellerden (augment yok → gerçekçi metrik)
        _write_samples('train', class_label, train_samples)
        _write_samples('val',   class_label, val_orig)

    print('\n─── Veri Seti Özeti ───')
    for s in ['train', 'val']:
        print(f'  {s:6s}: {written[s]} görsel')
    print(f'  Toplam: {sum(written.values())} görsel')
    print('\n⚠ Not: Görseller orijinal en-boy oranında; boyutlandırmayı YOLO letterbox ile yapar.')
    print('   Test split yok. Val yalnızca orijinal görsellerden; augment yalnızca train\'e uygulanır.')

## 7. Dataset YAML Yapılandırması

In [ ]:
import yaml

dataset_yaml_path = DATASET_DIR / 'dataset.yaml'

dataset_config = {
    'path':   str(DATASET_DIR.resolve()),
    'train':  'train/images',
    'val':    'val/images',
    'nc':     NUM_CLASSES,
    'names':  CLASS_NAMES,
}

with open(dataset_yaml_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print('dataset.yaml oluşturuldu:')
print(open(dataset_yaml_path).read())

In [ ]:
# Veri seti bütünlük kontrolü
errors = []
for split in ['train', 'val']:
    img_dir = DATASET_DIR / split / 'images'
    lbl_dir = DATASET_DIR / split / 'labels'
    imgs = list(img_dir.glob('*.jpg'))
    for img in imgs:
        lbl = lbl_dir / (img.stem + '.txt')
        if not lbl.exists():
            errors.append(f'Etiket eksik: {lbl}')
    print(f'{split:6s}: {len(imgs):4d} görsel | {len(list(lbl_dir.glob("*.txt"))):4d} etiket')

if errors:
    print(f'\n⚠ {len(errors)} hata bulundu:')
    for e in errors[:10]:
        print(f'  {e}')
else:
    print('\n✅ Tüm görsel-etiket çiftleri eksiksiz.')

## 8. Model Fine-Tuning

YOLOv11n ağırlıkları `yolo11n.pt` adıyla Ultralytics tarafından sağlanmaktadır.  
Eğitim, COCO pretrained ağırlıklar üzerinden transfer learning ile başlatılır.

In [ ]:
# YOLOv11n modelini yükle (pretrained)
model = YOLO('yolo11n.pt')
print(model.info())

In [ ]:
# ─── EĞİTİM BAŞLAT ────────────────────────────────────────────────────────────
results = model.train(
    # Veri & Model
    data         = str(dataset_yaml_path),
    model        = 'yolo11n.pt',
    pretrained   = True,

    # Eğitim parametreleri
    epochs       = EPOCHS,
    imgsz        = IMG_SIZE,
    batch        = BATCH_SIZE,
    workers      = WORKERS,

    # Optimizasyon
    optimizer    = 'SGD',        # 'Adam', 'AdamW', 'SGD'
    lr0          = LR0,
    lrf          = LRF,
    momentum     = MOMENTUM,
    weight_decay = WEIGHT_DECAY,
    warmup_epochs = 5,
    warmup_momentum = 0.8,
    warmup_bias_lr  = 0.1,

    # Regularizasyon
    dropout      = 0.1,
    label_smoothing = 0.1,

    # Loss ağırlıkları
    box          = 7.5,
    cls          = 0.5,
    dfl          = 1.5,

    # Yerleşik augmentation (YOLO internal)
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    degrees      = 15.0,
    translate    = 0.1,
    scale        = 0.5,
    shear        = 5.0,
    perspective  = 0.0005,
    flipud       = 0.1,
    fliplr       = 0.5,
    mosaic       = 1.0,
    mixup        = 0.2,
    copy_paste   = 0.2,
    erasing      = 0.4,
    close_mosaic = 15,    # son 15 epoch'ta mosaic'i kapat (daha stabil yakınsama)

    # Erken durdurma & kayıt
    patience     = PATIENCE,
    save         = True,
    save_period  = 10,
    project      = str(RUNS_DIR),
    name         = 'zerohalt_yolo11n',
    exist_ok     = True,

    # Performans
    amp          = True,     # Otomatik Karışık Hassasiyet
    cache        = True,     # Görselleri RAM'de önbelleğe al
    device       = 0 if torch.cuda.is_available() else 'cpu',
    seed         = SEED,
    deterministic = True,

    # Verbose
    verbose      = True,
    plots        = True,
)

print('\n✅ Eğitim tamamlandı!')
print(f'En iyi model: {RUNS_DIR}/zerohalt_yolo11n/weights/best.pt')

## 9. Eğitim Metrikleri Görselleştirme

In [ ]:
results_dir = Path(RUNS_DIR) / 'zerohalt_yolo11n'
csv_path    = results_dir / 'results.csv'

if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle('YOLOv11n Eğitim Metrikleri — ZeroHalt', fontsize=16, y=1.01)

    metric_pairs = [
        ('train/box_loss',  'val/box_loss',   'Box Loss'),
        ('train/cls_loss',  'val/cls_loss',   'Classification Loss'),
        ('train/dfl_loss',  'val/dfl_loss',   'DFL Loss'),
        ('metrics/precision(B)', None,        'Precision'),
        ('metrics/recall(B)',    None,        'Recall'),
        ('metrics/mAP50(B)',     None,        'mAP@0.50'),
        ('metrics/mAP50-95(B)',  None,        'mAP@0.50:0.95'),
        ('lr/pg0',               None,        'Learning Rate (pg0)'),
    ]

    ax_list = axes.flatten()
    for i, (train_col, val_col, title) in enumerate(metric_pairs):
        ax = ax_list[i]
        if train_col in df.columns:
            ax.plot(df['epoch'], df[train_col], label='train', color='#2196F3')
        if val_col and val_col in df.columns:
            ax.plot(df['epoch'], df[val_col], label='val', color='#F44336', linestyle='--')
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    ax_list[-1].axis('off')  # 9. hücre boş
    plt.tight_layout()
    plt.savefig('training_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Son epoch metrikleri:')
    print(df.tail(1).T)
else:
    print('results.csv bulunamadı. Önce eğitimi tamamlayın.')

In [ ]:
# Confusion matrix görüntüle
cm_path = results_dir / 'confusion_matrix_normalized.png'
if not cm_path.exists():
    cm_path = results_dir / 'confusion_matrix.png'
if cm_path.exists():
    img = Image.open(cm_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.title('Confusion Matrix', fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Confusion matrix henüz oluşturulmadı.')

## 10. Model Değerlendirme (Validation & Test)

In [ ]:
best_model_path = results_dir / 'weights' / 'best.pt'
best_model      = YOLO(str(best_model_path))

# Validation seti değerlendirme
val_metrics = best_model.val(
    data    = str(dataset_yaml_path),
    split   = 'val',
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    device  = 0 if torch.cuda.is_available() else 'cpu',
    conf    = 0.001,
    iou     = 0.6,
    verbose = True,
)

print('\n─── Validation Sonuçları ───')
print(f'mAP@0.50:      {val_metrics.box.map50:.4f}')
print(f'mAP@0.50:0.95: {val_metrics.box.map:.4f}')
print(f'Precision:     {val_metrics.box.mp:.4f}')
print(f'Recall:        {val_metrics.box.mr:.4f}')

In [ ]:
# Sınıf bazlı AP raporu (val_metrics üzerinden — test split yok)
ap_per_class = val_metrics.box.ap_class_index
ap50_vals    = val_metrics.box.ap50
ap_vals      = val_metrics.box.ap

df_ap = pd.DataFrame({
    'class':      [CLASS_NAMES[i] for i in ap_per_class],
    'AP@0.50':    ap50_vals,
    'AP@0.50:95': ap_vals,
})
print('\n─── Sınıf Bazlı AP ───')
print(df_ap.to_string(index=False))

# Sınıf bazlı AP grafiği
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_ap))
w = 0.35
ax.bar(x - w/2, df_ap['AP@0.50'],    width=w, label='AP@0.50',    color='#2196F3')
ax.bar(x + w/2, df_ap['AP@0.50:95'], width=w, label='AP@0.50:95', color='#FF9800')
ax.set_xticks(x)
ax.set_xticklabels(df_ap['class'], rotation=15)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Average Precision')
ax.set_title('Sınıf Bazlı Average Precision', fontsize=13)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('ap_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Inference & Görsel Test

In [ ]:
# Val veri setinden rastgele görseller üzerinde inference (test split yok)
val_img_dir  = DATASET_DIR / 'val' / 'images'
val_images   = list(val_img_dir.glob('*.jpg'))
sample_tests = random.sample(val_images, min(9, len(val_images)))

CONF_THRESH = 0.25
IOU_THRESH  = 0.45

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle(f'Val Inference (conf={CONF_THRESH}, iou={IOU_THRESH})', fontsize=14)

CLASS_COLORS = [
    (33, 150, 243), (76, 175, 80), (255, 152, 0), (233, 30, 99), (156, 39, 176)
]

for ax, img_path in zip(axes.flatten(), sample_tests):
    preds = best_model.predict(
        source  = str(img_path),
        conf    = CONF_THRESH,
        iou     = IOU_THRESH,
        verbose = False,
    )[0]

    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w    = img_rgb.shape[:2]

    for box in preds.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls_id = int(box.cls[0])
        conf   = float(box.conf[0])
        color  = CLASS_COLORS[cls_id % len(CLASS_COLORS)]
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
        label  = f'{CLASS_NAMES[cls_id]} {conf:.2f}'
        cv2.putText(img_rgb, label, (x1, max(y1 - 8, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

    ax.imshow(img_rgb)
    ax.set_title(img_path.stem[:30], fontsize=8)
    ax.axis('off')

for ax in axes.flatten()[len(sample_tests):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig('inference_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tek görsel inference (özel görsel için)
def predict_single(image_path: str, conf: float = 0.25) -> None:
    preds  = best_model.predict(source=image_path, conf=conf, verbose=False)[0]
    img    = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)

    for box in preds.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls_id = int(box.cls[0])
        conf_v = float(box.conf[0])
        color  = CLASS_COLORS[cls_id % len(CLASS_COLORS)]
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
        cv2.putText(img, f'{CLASS_NAMES[cls_id]} {conf_v:.2f}',
                    (x1, max(y1 - 10, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.title(f'Inference: {Path(image_path).name}', fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'{len(preds.boxes)} nesne tespit edildi.')

# Kullanım: predict_single('parcalar/piston/images.jpeg')
predict_single(str(sample_tests[0]))

## 12. Hiperparametre Araştırması (Opsiyonel)

Ultralytics yerleşik `tune()` ile otomatik hiperparametre optimizasyonu yapılabilir.

In [ ]:
RUN_TUNING = False   # ← True yaparsanız ~30 iterasyonluk tune çalışır

if RUN_TUNING:
    tune_model = YOLO('yolo11n.pt')
    tune_results = tune_model.tune(
        data       = str(dataset_yaml_path),
        epochs     = 30,
        iterations = 30,
        imgsz      = IMG_SIZE,
        batch      = BATCH_SIZE,
        optimizer  = 'AdamW',
        plots      = True,
        save       = True,
        device     = 0 if torch.cuda.is_available() else 'cpu',
    )
    print('En iyi hiperparametreler:', tune_results)
else:
    print('Hiperparametre araştırması devre dışı.')

## 13. Model Export

In [ ]:
# ONNX export (CPU/Edge deployment)
onnx_path = best_model.export(
    format   = 'onnx',
    imgsz    = IMG_SIZE,
    opset    = 17,
    dynamic  = False,
    simplify = True,
)
print(f'ONNX modeli: {onnx_path}')

In [ ]:
# TorchScript export
torchscript_path = best_model.export(
    format = 'torchscript',
    imgsz  = IMG_SIZE,
)
print(f'TorchScript modeli: {torchscript_path}')

In [ ]:
# TensorRT export (NVIDIA GPU'da çalışır)
RUN_TRT = torch.cuda.is_available()

if RUN_TRT:
    trt_path = best_model.export(
        format    = 'engine',
        imgsz     = IMG_SIZE,
        half      = True,    # FP16
        device    = 0,
        workspace = 4,       # GB
    )
    print(f'TensorRT engine: {trt_path}')
else:
    print('TensorRT export için NVIDIA GPU gereklidir.')

## 14. Model Özeti & Deployment Rehberi

In [ ]:
# Eğitilen modelin özet raporu
print('=' * 60)
print('ZeroHalt YOLOv11n — Model Özeti')
print('=' * 60)
print(f'Sınıflar    : {CLASS_NAMES}')
print(f'Görsel boyut: {IMG_SIZE}x{IMG_SIZE}')
print(f'Epoch sayısı: {EPOCHS}')
print(f'Batch size  : {BATCH_SIZE}')
print()
print('Validation Metrikleri:')
print(f'  mAP@0.50      : {val_metrics.box.map50:.4f}')
print(f'  mAP@0.50:0.95 : {val_metrics.box.map:.4f}')
print(f'  Precision     : {val_metrics.box.mp:.4f}')
print(f'  Recall        : {val_metrics.box.mr:.4f}')
print()
print('Model Dosyaları:')
print(f'  Best PT  : {best_model_path}')
print(f'  ONNX     : {onnx_path}')
print('=' * 60)

In [ ]:
# Flask / FastAPI ile deployment için hazır inference kodu
DEPLOY_CODE = '''
from ultralytics import YOLO
from pathlib import Path

CLASS_NAMES = ["alternator", "brake_disc", "engine_control",
               "hydraulic_filter", "piston"]

model = YOLO("runs/zerohalt_yolo11n/weights/best.pt")

def detect_parts(image_path: str, conf: float = 0.25) -> list[dict]:
    """Görsel üzerinde parça tespiti yapar."""
    results = model.predict(source=image_path, conf=conf, verbose=False)[0]
    detections = []
    for box in results.boxes:
        x1, y1, x2, y2 = map(float, box.xyxy[0].tolist())
        detections.append({
            "class_id":   int(box.cls[0]),
            "class_name": CLASS_NAMES[int(box.cls[0])],
            "confidence": round(float(box.conf[0]), 4),
            "bbox":       [x1, y1, x2, y2],
        })
    return detections

# Kullanım:
# detections = detect_parts("parcalar/piston/images.jpeg")
# print(detections)
'''
print('Deployment kodu:')
print(DEPLOY_CODE)

In [ ]:
# Deployment kodunu dosyaya kaydet
deploy_path = BASE_DIR / 'src' / 'yolo_inference.py'
deploy_path.parent.mkdir(parents=True, exist_ok=True)
deploy_path.write_text(DEPLOY_CODE.strip())
print(f'Inference modülü kaydedildi: {deploy_path}')

---

## Sonuç

| Adım | Durum |
|------|-------|
| Ortam kurulumu | ✅ |
| Veri keşfi & analiz | ✅ |
| Augmentation pipeline | ✅ |
| YOLO veri seti oluşturma | ✅ |
| YOLOv11n fine-tuning | ✅ |
| Metrik görselleştirme | ✅ |
| Val / Test değerlendirme | ✅ |
| Inference & görsel test | ✅ |
| ONNX / TRT export | ✅ |
| Deployment modülü | ✅ |

**Öneriler:**
- Sınıf başı görsel sayısını 300+ 'a çıkarmak modeli önemli ölçüde güçlendirir.
- `engine control` ve `hydroic filter` sınıflarına özellikle çeşitli arka plan görselleri ekleyin.
- mAP@0.50 < 0.75 ise annotation kalitesini kontrol edin, `label_smoothing` değerini düşürün.
- Üretime almadan önce gerçek sahne koşullarında (farklı ışık, açı) test yapın.